# Notebook 2: Fine-Tuning and Forgetting Measurement

This notebook:
1. Loads tokenized datasets and pretrained DistilGPT-2
2. Evaluates baseline performance on sentiment and BoolQ tasks
3. Fine-tunes DistilGPT-2 on sentiment classification
4. Evaluates post-fine-tuning performance
5. Computes forgetting metric: baseline_boolq - finetuned_boolq

In [ ]:
# Import required libraries
import os
import json
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from tqdm import tqdm

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Libraries imported successfully!")

## 1. Load Datasets and Model

In [ ]:
# Load tokenized datasets
print("Loading tokenized datasets...")
tokenized_train = load_from_disk("data/tokenized_sentiment_train")
tokenized_test = load_from_disk("data/tokenized_sentiment_test")

# Load BoolQ eval set
with open("data/boolq_eval.json", "r") as f:
    boolq_eval = json.load(f)

print(f"Train set size: {len(tokenized_train)}")
print(f"Test set size: {len(tokenized_test)}")
print(f"BoolQ eval set size: {len(boolq_eval)}")

In [ ]:
# Load pretrained DistilGPT-2 model and tokenizer
model_name = "distilgpt2"
print(f"Loading model and tokenizer: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

print(f"Model loaded. Vocab size: {tokenizer.vocab_size}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Baseline Evaluation Functions

In [ ]:
def normalize_prediction(pred_text):
    """Normalize prediction for sentiment comparison."""
    pred_text = pred_text.strip().lower()
    # Take first token before any space
    pred_text = pred_text.split()[0] if pred_text else ""
    return pred_text

def evaluate_sentiment_accuracy(model, tokenizer, test_dataset, labels, max_samples=None):
    """
    Evaluate sentiment classification accuracy using generation.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        test_dataset: Tokenized test dataset
        labels: Dictionary mapping label IDs to label strings
        max_samples: Maximum number of samples to evaluate (None for all)
        
    Returns:
        accuracy: Classification accuracy
        predictions: List of (predicted, actual) tuples
    """
    model.eval()
    correct = 0
    total = 0
    predictions = []
    
    # Create reverse label mapping (string -> id)
    label_to_id = {v: k for k, v in labels.items()}
    
    # Extract label strings
    label_strings = list(labels.values())
    
    eval_size = min(max_samples, len(test_dataset)) if max_samples else len(test_dataset)
    
    with torch.no_grad():
        for i in tqdm(range(eval_size), desc="Evaluating sentiment"):
            # Get the input_ids (which contains the full prompt + target)
            input_ids = torch.tensor(test_dataset[i]['input_ids']).unsqueeze(0)
            
            # Decode to get the original prompt
            full_text = tokenizer.decode(input_ids[0], skip_special_tokens=False)
            
            # Extract the prompt part (everything before the answer)
            # The prompt should end with "Answer:"
            if "Answer:" in full_text:
                prompt_text = full_text.split("Answer:")[0] + "Answer:"
            else:
                # Fallback: reconstruct prompt from input
                prompt_text = full_text
            
            # Get the actual label (last token/word after Answer:)
            if "Answer:" in full_text:
                actual_label = full_text.split("Answer:")[-1].strip()
                actual_label = normalize_prediction(actual_label)
            else:
                actual_label = None
            
            # Tokenize the prompt
            prompt_ids = tokenizer(prompt_text, return_tensors="pt")['input_ids']
            
            # Generate prediction
            output = model.generate(
                prompt_ids,
                max_new_tokens=3,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
            
            # Decode only the new tokens
            generated_text = tokenizer.decode(output[0][prompt_ids.shape[1]:], skip_special_tokens=True)
            pred_label = normalize_prediction(generated_text)
            
            # Compare prediction with actual label
            # Check if prediction matches any label string (first few chars)
            if actual_label:
                # Find best matching label
                matches = [label for label in label_strings if label.startswith(pred_label) or pred_label.startswith(label[:3])]
                if matches:
                    pred_label = matches[0]
                
                # Compare
                if pred_label == actual_label or (pred_label in actual_label) or (actual_label in pred_label):
                    correct += 1
                
                predictions.append((pred_label, actual_label))
                total += 1
    
    accuracy = correct / total if total > 0 else 0.0
    return accuracy, predictions

In [ ]:
def evaluate_boolq_accuracy(model, tokenizer, boolq_eval):
    """
    Evaluate BoolQ accuracy using language model scoring.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        boolq_eval: List of BoolQ evaluation examples
        
    Returns:
        accuracy: Classification accuracy
        predictions: List of (predicted, actual) tuples
    """
    model.eval()
    correct = 0
    predictions = []
    
    candidate_labels = ["yes", "no"]
    
    with torch.no_grad():
        for example in tqdm(boolq_eval, desc="Evaluating BoolQ"):
            prompt = example["prompt"]
            actual_answer = example["answer"]
            
            # Score each candidate label
            label_scores = {}
            
            for label in candidate_labels:
                # Build full text: prompt + label
                full_text = prompt + " " + label
                
                # Tokenize
                inputs = tokenizer(full_text, return_tensors="pt")
                input_ids = inputs['input_ids']
                
                # Get logits
                outputs = model(input_ids)
                logits = outputs.logits[0]  # Shape: (seq_len, vocab_size)
                
                # Compute negative log-likelihood per token
                # We only score the label tokens (not the prompt)
                prompt_len = len(tokenizer(prompt, return_tensors="pt")['input_ids'][0])
                label_tokens = input_ids[0][prompt_len:]
                
                if len(label_tokens) > 0:
                    # Get log probabilities for the label tokens
                    log_probs = torch.log_softmax(logits, dim=-1)
                    
                    # Sum log probabilities for label tokens
                    total_log_prob = 0.0
                    for i, token_id in enumerate(label_tokens):
                        if prompt_len + i < len(log_probs):
                            total_log_prob += log_probs[prompt_len + i - 1, token_id].item()
                    
                    label_scores[label] = total_log_prob
                else:
                    label_scores[label] = float('-inf')
            
            # Pick label with highest score (lowest negative log-likelihood)
            pred_answer = max(label_scores, key=label_scores.get)
            
            if pred_answer == actual_answer:
                correct += 1
            
            predictions.append((pred_answer, actual_answer))
    
    accuracy = correct / len(boolq_eval) if len(boolq_eval) > 0 else 0.0
    return accuracy, predictions

## 3. Baseline Evaluation

In [ ]:
# Evaluate baseline sentiment accuracy
print("\n" + "="*50)
print("BASELINE EVALUATION")
print("="*50)

labels = {0: "negative", 1: "neutral", 2: "positive"}

print("\nEvaluating baseline sentiment accuracy...")
baseline_sentiment_acc, baseline_sentiment_preds = evaluate_sentiment_accuracy(
    model, tokenizer, tokenized_test, labels, max_samples=100
)
print(f"Baseline Sentiment Accuracy: {baseline_sentiment_acc:.4f} ({baseline_sentiment_acc*100:.2f}%)")

In [ ]:
# Evaluate baseline BoolQ accuracy
print("\nEvaluating baseline BoolQ accuracy...")
baseline_boolq_acc, baseline_boolq_preds = evaluate_boolq_accuracy(
    model, tokenizer, boolq_eval
)
print(f"Baseline BoolQ Accuracy: {baseline_boolq_acc:.4f} ({baseline_boolq_acc*100:.2f}%)")
print(f"\nBaseline BoolQ Predictions:")
for i, (pred, actual) in enumerate(baseline_boolq_preds):
    print(f"  Example {i+1}: Predicted={pred}, Actual={actual}")

## 4. Fine-Tuning

In [ ]:
# Load a fresh pretrained model for fine-tuning
print("\n" + "="*50)
print("FINE-TUNING")
print("="*50)

print("Loading fresh pretrained model for fine-tuning...")
ft_model = AutoModelForCausalLM.from_pretrained(model_name)
ft_tokenizer = AutoTokenizer.from_pretrained(model_name)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token

print("Model loaded. Starting fine-tuning...")

In [ ]:
# Set up training arguments
training_args = TrainingArguments(
    output_dir="models/distilgpt2_sentiment_ft",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="no",
    report_to=[],
    seed=42
)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=ft_tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# Initialize trainer
trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator
)

print("Trainer initialized. Starting training...")
trainer.train()
print("\nFine-tuning completed!")

In [ ]:
# Save the fine-tuned model
print("Saving fine-tuned model...")
ft_model.save_pretrained("models/distilgpt2_sentiment_ft")
ft_tokenizer.save_pretrained("models/distilgpt2_sentiment_ft")
print("Model saved to: models/distilgpt2_sentiment_ft")

## 5. Post-Fine-Tuning Evaluation

In [ ]:
# Evaluate fine-tuned sentiment accuracy
print("\n" + "="*50)
print("POST-FINE-TUNING EVALUATION")
print("="*50)

print("\nEvaluating fine-tuned sentiment accuracy...")
finetuned_sentiment_acc, finetuned_sentiment_preds = evaluate_sentiment_accuracy(
    ft_model, ft_tokenizer, tokenized_test, labels, max_samples=100
)
print(f"Fine-Tuned Sentiment Accuracy: {finetuned_sentiment_acc:.4f} ({finetuned_sentiment_acc*100:.2f}%)")

In [ ]:
# Evaluate fine-tuned BoolQ accuracy
print("\nEvaluating fine-tuned BoolQ accuracy...")
finetuned_boolq_acc, finetuned_boolq_preds = evaluate_boolq_accuracy(
    ft_model, ft_tokenizer, boolq_eval
)
print(f"Fine-Tuned BoolQ Accuracy: {finetuned_boolq_acc:.4f} ({finetuned_boolq_acc*100:.2f}%)")
print(f"\nFine-Tuned BoolQ Predictions:")
for i, (pred, actual) in enumerate(finetuned_boolq_preds):
    print(f"  Example {i+1}: Predicted={pred}, Actual={actual}")

## 6. Compute Forgetting Metric

In [ ]:
# Compute forgetting metric
forgetting = baseline_boolq_acc - finetuned_boolq_acc

print("\n" + "="*50)
print("FORGETTING SUMMARY")
print("="*50)
print(f"\nBaseline Sentiment Accuracy:    {baseline_sentiment_acc:.4f} ({baseline_sentiment_acc*100:.2f}%)")
print(f"Fine-Tuned Sentiment Accuracy:   {finetuned_sentiment_acc:.4f} ({finetuned_sentiment_acc*100:.2f}%)")
print(f"Sentiment Improvement:           {finetuned_sentiment_acc - baseline_sentiment_acc:.4f} ({((finetuned_sentiment_acc - baseline_sentiment_acc)*100):.2f}%)")
print(f"\nBaseline BoolQ Accuracy:        {baseline_boolq_acc:.4f} ({baseline_boolq_acc*100:.2f}%)")
print(f"Fine-Tuned BoolQ Accuracy:       {finetuned_boolq_acc:.4f} ({finetuned_boolq_acc*100:.2f}%)")
print(f"Forgetting (BoolQ):              {forgetting:.4f} ({forgetting*100:.2f}%)")
print(f"\nInterpretation:")
if forgetting > 0:
    print(f"  ✓ Model forgot BoolQ knowledge (decreased by {forgetting*100:.2f}%)")
else:
    print(f"  ✓ Model retained/gained BoolQ knowledge (changed by {forgetting*100:.2f}%)")

In [ ]:
# Save results to JSON
results = {
    "baseline_sentiment": float(baseline_sentiment_acc),
    "baseline_boolq": float(baseline_boolq_acc),
    "finetuned_sentiment": float(finetuned_sentiment_acc),
    "finetuned_boolq": float(finetuned_boolq_acc),
    "forgetting": float(forgetting)
}

with open("data/results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nResults saved to: data/results.json")
print(json.dumps(results, indent=2))

## Summary

This notebook has:
- ✅ Evaluated baseline performance on sentiment and BoolQ tasks
- ✅ Fine-tuned DistilGPT-2 on sentiment classification
- ✅ Evaluated post-fine-tuning performance
- ✅ Computed forgetting metric
- ✅ Saved model and results

**Next Steps:** Proceed to Notebook 3 for visualizations and analysis.